# 08 - CDC-anchored season severity tiers, D2, and the severity classifier

Implements `PLAN-severity-tiers.md`, which was locked by grill and approved by Codex at round 3
(`PLAN-REVIEW-LOG-severity-tiers.md`). Acts on Dr. Mitra's 2026-08-02 ruling to anchor severity
tiers to CDC's published intensity thresholds.

**What this notebook does NOT claim.** The tiers here are an **ILI-only approximation** of CDC's
season severity framework, built from CDC's published ILI intensity thresholds. They are not CDC's
official season severity classification, which is a 2-of-3 indicator vote across ILI, hospitalization
rate, and pneumonia-and-influenza mortality. This project has ILI from 2003, hospitalization only
from 2009, and no mortality data.

**Status: the tier definition is a target definition and is pending Joshua's checkpoint-2 review.**
Dr. Mitra approved the method, not these specific numbers.

## Setup, and the CDC thresholds as cited constants

The three threshold values appear in exactly one place. The framework and tier names come from CDC's
severity-assessment documentation; the numeric values come from Biggerstaff 2018 and are that paper's
historical values, not CDC's present operational ones. Those are cited separately on purpose.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score

DATA_DIR = next((Path(p) for p in ["data/raw", "../data/raw"] if Path(p).exists()), Path("data/raw"))
RESULTS_DIR = DATA_DIR.parent.parent / "results"; RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = DATA_DIR.parent.parent / "figures"; FIG_DIR.mkdir(exist_ok=True)
EXCLUDED = {"2008-09", "2009-10", "2020-21"}
DECISION_WEEKS = [8, 12, 16]
SEED = 42
BOOTSTRAP_N = 20000
np.random.seed(SEED)

CDC_ILI_IT = {"IT50": 4.4, "IT90": 6.6, "IT98": 8.6}
CDC_IT_SOURCE = "Biggerstaff M, et al. Am J Epidemiol. 2018;187(5):1040-1050. doi:10.1093/aje/kwx334"
CDC_IT_REFERENCE_SEASONS = "2003-04 through 2014-15, excluding the 2009 pandemic"
CDC_IT_VINTAGE_NOTE = ("2018 published values. CDC's current operational values are not pinned here "
                       "and are not claimed to differ; using the published historical values is a "
                       "deliberate methodological choice, made for citability.")
NOT_ASSESSED = {"2020-21": "CDC did not assign a severity classification; minimal influenza "
                           "activity under COVID-19 mitigations"}
CDC_PUBLISHED = {"2003-04": "High", "2004-05": "Moderate", "2005-06": "Low", "2006-07": "Low",
                 "2007-08": "Moderate", "2008-09": "Low", "2009-10": "Moderate",
                 "2010-11": "Moderate", "2011-12": "Low", "2012-13": "Moderate",
                 "2013-14": "Moderate", "2014-15": "High"}
NON_COMPARABLE = {
    "2009-10": "the 2009 H1N1 pandemic season itself",
    "2008-09": "pandemic-adjacent: this project's MMWR season runs to wk39 and absorbs the spring "
               "2009 H1N1 wave, which CDC did not fold into its 2008-09 figure",
}
CLASSES = ["Low", "Moderate", "High"]
RF_PARAMS = dict(random_state=SEED, n_estimators=500, min_samples_leaf=2, max_depth=None,
                 class_weight="balanced", max_features="sqrt")


def tier_of(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if v < CDC_ILI_IT["IT50"]: return "Low"
    if v < CDC_ILI_IT["IT90"]: return "Moderate"
    if v < CDC_ILI_IT["IT98"]: return "High"
    return "Very High"


def season_of(y, w):
    sy = y if w >= 40 else y - 1
    return f"{sy}-{str(sy + 1)[2:]}", sy


def sw(w):
    return w - 39 if w >= 40 else w + 13


def _load_ilinet(path, region_filter=None):
    d = pd.read_csv(path, skiprows=1, na_values=["X"])
    _i = d.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
    d["season"] = [x[0] for x in _i]; d["ssy"] = [x[1] for x in _i]
    d["order"] = d["WEEK"].apply(lambda w: w if w >= 40 else w + 100)
    d["sw"] = d["WEEK"].apply(sw)
    d["% WEIGHTED ILI"] = pd.to_numeric(d["% WEIGHTED ILI"], errors="coerce")
    return d.sort_values(["ssy", "order"]).reset_index(drop=True)


## Notebook 02 targets, plus the new classification statistic

`peak_ili_pct` (maximum of a 3-week centered smoother) is reconstructed exactly as in 05/06/07 and is
**left untouched**. This notebook adds a parallel variable, `season_peak_max`, the season's peak weekly
raw `% WEIGHTED ILI`.

**Why the raw maximum and not the geometric mean of the top three weeks.** The geometric mean of the
highest values is how the Moving Epidemic Method *estimates the thresholds* from a pool of reference
seasons. The quantity a season is then *classified* on is where its indicator peaked. Using the
geometric mean to classify conflates the two. `season_peak_gm3` is retained only to document the
discarded alternative.

**2020-21 gets no tier.** CDC did not assess it, so assigning one would be this project's judgement
wearing CDC's label.

In [ ]:
ili = _load_ilinet(DATA_DIR / "ILINet.csv")
assert set(ili["REGION TYPE"].unique()) == {"National"}, "national file expected here"


def _complete(g):
    sy = int(g["ssy"].iloc[0])
    sp = sorted(g.loc[g["YEAR"] == sy, "WEEK"]); ep = sorted(g.loc[g["YEAR"] == sy + 1, "WEEK"])
    return bool(sp and sp[0] == 40 and ep and ep[0] == 1 and ep[-1] == 39)


COMPLETE = [s for s, g in ili.groupby("season") if _complete(g)]
weekly = ili[ili["season"].isin(COMPLETE)].copy()

rows = []
for s, g in weekly.groupby("season"):
    g = g.sort_values("order")
    sm3 = g["% WEIGHTED ILI"].rolling(3, center=True).mean()
    sm5 = g["% WEIGHTED ILI"].rolling(5, center=True).mean()
    raw = g["% WEIGHTED ILI"]
    rows.append(dict(season=s, ssy=int(g["ssy"].iloc[0]),
                     peak_week=int(g.loc[sm3.idxmax(), "WEEK"]),
                     peak_ili_pct=round(float(sm3.max()), 3),
                     peak_week_sm5=int(g.loc[sm5.idxmax(), "WEEK"]),
                     season_peak_max=round(float(raw.max()), 3),
                     season_peak_max_week=int(g.loc[raw.idxmax(), "WEEK"]),
                     season_peak_gm3=round(float(np.exp(np.log(raw.nlargest(3)).mean())), 4)))
season_table = pd.DataFrame(rows).sort_values("ssy").reset_index(drop=True)
season_table["fragile_peak_week"] = season_table["peak_week"] != season_table["peak_week_sm5"]
EVAL = [s for s in season_table["season"] if s not in EXCLUDED]
assert len(EVAL) == 19 and len(season_table) == 22

season_table["tier_max"] = season_table["season_peak_max"].apply(tier_of)
season_table["tier_smoothed"] = season_table["peak_ili_pct"].apply(tier_of)
season_table["tier_gm3"] = season_table["season_peak_gm3"].apply(tier_of)
season_table["tier_margin"] = season_table["season_peak_max"].apply(
    lambda v: round(min(abs(v - t) for t in CDC_ILI_IT.values()), 4))
season_table["tier_variants_disagree"] = season_table.apply(
    lambda r: len({r["tier_max"], r["tier_smoothed"], r["tier_gm3"]}) > 1, axis=1)
season_table["tier_boundary_sensitive"] = (
    (season_table["tier_margin"] < 0.30) | season_table["tier_variants_disagree"])
season_table["tier_assigned"] = season_table.apply(
    lambda r: "Not assessed" if r["season"] in NOT_ASSESSED else r["tier_max"], axis=1)
season_table["excluded_from_modeling"] = season_table["season"].isin(EXCLUDED)

assessed = season_table[~season_table["season"].isin(NOT_ASSESSED)]
model_tbl = season_table[season_table["season"].isin(EVAL)].reset_index(drop=True)
assert int((season_table["tier_max"] == "Very High").sum()) == 0


## Validation against CDC's published season classifications

CDC published overall severity for 2003-04 through 2014-15 in the same paper. Both agreement figures
are reported together, overall first. Quoting only the comparable-season figure would overstate
agreement, because the two excluded seasons are two of the three disagreements.

2009-10 is the pandemic season itself. 2008-09 is pandemic-adjacent: this project's MMWR season runs
to wk39 and therefore absorbs the spring 2009 H1N1 wave, which CDC did not fold into its 2008-09
figure. That is a season-boundary difference, not a framework difference.

In [ ]:
# ---------------------------------------------------------------- CDC validation
val = pd.DataFrame([
    dict(season=s, cdc_published=t, ili_only_tier=r["tier_max"],
         season_peak_max=r["season_peak_max"], agrees=bool(r["tier_max"] == t),
         comparable=s not in NON_COMPARABLE, non_comparable_reason=NON_COMPARABLE.get(s, ""))
    for s, t in CDC_PUBLISHED.items()
    for r in [season_table[season_table["season"] == s].iloc[0]]])
agree_all, n_all = int(val["agrees"].sum()), len(val)
comp = val[val["comparable"]]
agree_comp, n_comp = int(comp["agrees"].sum()), len(comp)


## Through-W features and the index firewall

Primary features are ILI-derived only and near-real-time under the project's index-cutoff convention.
ILINet is itself revised after publication, so "near-real-time" is the accurate claim; the substantive
contrast is with strain, hospitalization and vaccine coverage, which are materially lag-reported or
survey-revised. Strain is loaded here for the **retrospective secondary panel only**.

The firewall governs **features**, not the target. The label derives from the full-season peak by
design, exactly as in 05 and 06 where the target is also the realized season peak.

In [ ]:
# ---------------------------------------------------------------- features
weekly_lag = weekly.sort_values(["season", "order"]).drop_duplicates(["season", "sw"], keep="last")
REALTIME_COLS = ["cum_ili", "ili_lag_1", "ili_lag_2", "ili_lag_3", "ili_lag_4", "ili_rolling4"]


def cum_ili_thruW(s, W):
    return float(weekly[(weekly["season"] == s) & (weekly["sw"] <= W)]["% WEIGHTED ILI"].sum())


def ili_lags(s, W):
    d = weekly_lag[weekly_lag["season"] == s].set_index("sw")["% WEIGHTED ILI"]
    return [float(d.loc[W - (k - 1)]) for k in range(1, 5)]


# Strain, for the RETROSPECTIVE secondary panel only. Established 2015-16 stitch, as in 06/07.
def load_nrevss(f):
    d = pd.read_csv(DATA_DIR / f, skiprows=1, na_values=["X", "XX"])
    ii = d.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
    d["season"] = [x[0] for x in ii]; d["ssy"] = [x[1] for x in ii]
    d["sw"] = d["WEEK"].apply(sw)
    return d


def strain_buckets(d):
    col = lambda n: d[n] if n in d.columns else 0
    return pd.DataFrame({"season": d["season"], "ssy": d["ssy"], "sw": d["sw"],
                         "A(H1N1)": col("A (H1)") + col("A (2009 H1N1)"),
                         "A(H3N2)": col("A (H3)"),
                         "B": col("B") + col("BVic") + col("BYam")})


_comb = strain_buckets(load_nrevss("ICL_NREVSS_Combined_prior_to_2015_16.csv"))
_phl = strain_buckets(load_nrevss("ICL_NREVSS_Public_Health_Labs.csv"))
strain_wk = pd.concat([_comb[_comb["ssy"] <= 2014], _phl[_phl["ssy"] >= 2015]]).reset_index(drop=True)


def dominant_strain_thruW(s, W):
    d = strain_wk[(strain_wk["season"] == s) & (strain_wk["sw"] <= W)]
    tot = d[["A(H1N1)", "A(H3N2)", "B"]].sum()
    return "none" if tot.sum() <= 0 else str(tot.idxmax())


def feat_df(W, seasons):
    peak_map = dict(zip(season_table["season"], season_table["season_peak_max"]))
    tier_map = dict(zip(season_table["season"], season_table["tier_max"]))
    out = []
    for s in seasons:
        lags = ili_lags(s, W)
        out.append(dict(season=s, cum_ili=cum_ili_thruW(s, W), ili_lag_1=lags[0], ili_lag_2=lags[1],
                        ili_lag_3=lags[2], ili_lag_4=lags[3], ili_rolling4=float(np.mean(lags)),
                        strain=dominant_strain_thruW(s, W),
                        season_peak_max=float(peak_map[s]), tier=tier_map[s]))
    return pd.DataFrame(out)


audit_pairs = 0
for W in DECISION_WEEKS:
    for s in season_table["season"]:
        used = weekly[(weekly["season"] == s) & (weekly["sw"] <= W)]
        assert int(used["sw"].max()) <= W
        assert max(W - (k - 1) for k in range(1, 5)) <= W
        assert min(W - (k - 1) for k in range(1, 5)) >= 1
        audit_pairs += 1


## Classifier and baselines

**Three classes.** Very High is empty across all 22 seasons, so a 4-class model would carry a class it
can never predict.

**Baselines before models**, per working-agreement checkpoint 3. `B3`, regression-then-threshold, is
the baseline that matters: this project's one honest positive result is a one-variable regression, and
a classifier that cannot beat discretizing it has earned nothing. `B1` draws from each training fold's
class distribution, never the full sample, which would leak the held-out label distribution.

Hyperparameters, seed and scoring rule were fixed in the plan before any result was seen.

In [ ]:
def macro_f1(yt, yp):
    return float(f1_score(yt, yp, labels=CLASSES, average="macro", zero_division=0))


def run_W(W, cols, seasons=EVAL, use_strain=False):
    df = feat_df(W, seasons)
    X = df[cols].to_numpy(float)
    if use_strain:
        X = np.hstack([X, pd.get_dummies(df["strain"]).reindex(
            columns=["A(H1N1)", "A(H3N2)", "B", "none"], fill_value=0).to_numpy(float)])
    y = df["tier"].to_numpy(); peaks = df["season_peak_max"].to_numpy(float); n = len(df)
    pred_rf, pred_b0, pred_b2, pred_b3 = [], [], [], []
    b1 = np.empty((2000, n), dtype=object); rng_b1 = np.random.default_rng(SEED)
    for i in range(n):
        tr = np.arange(n) != i
        vals, cnts = np.unique(y[tr], return_counts=True)
        pred_b0.append(vals[np.argmax(cnts)])
        b1[:, i] = rng_b1.choice(vals, size=2000, p=cnts / cnts.sum())
        pred_b2.append(tier_of(float(peaks[tr].mean())))
        sl, ic = np.polyfit(df[cols[0]].to_numpy(float)[tr], peaks[tr], 1)
        pred_b3.append(tier_of(float(sl * df[cols[0]].to_numpy(float)[i] + ic)))
        rf = RandomForestClassifier(**RF_PARAMS); rf.fit(X[tr], y[tr])
        pred_rf.append(rf.predict(X[i:i + 1])[0])
    pred_rf, pred_b3 = np.array(pred_rf), np.array(pred_b3)
    rng = np.random.default_rng(SEED); idx = rng.integers(0, n, size=(BOOTSTRAP_N, n))
    rf_boot = [macro_f1(y[i], pred_rf[i]) for i in idx]
    diff = [macro_f1(y[i], pred_rf[i]) - macro_f1(y[i], pred_b3[i]) for i in idx]
    lo, hi = float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))
    return dict(
        W=W, n=n, class_counts={c: int((y == c).sum()) for c in CLASSES},
        confusion_matrix={"labels": CLASSES,
                          "rows_true_cols_pred": confusion_matrix(y, pred_rf, labels=CLASSES).tolist()},
        per_class_f1={c: round(float(v), 4) for c, v in
                      zip(CLASSES, f1_score(y, pred_rf, labels=CLASSES, average=None, zero_division=0))},
        accuracy=round(float(accuracy_score(y, pred_rf)), 4),
        macro_f1={"RF": round(macro_f1(y, pred_rf), 4),
                  "B0_majority": round(macro_f1(y, pred_b0), 4),
                  "B1_stratified_random_mean": round(float(np.mean(
                      [macro_f1(y, b1[r]) for r in range(b1.shape[0])])), 4),
                  "B2_climatology_threshold": round(macro_f1(y, pred_b2), 4),
                  "B3_regression_then_threshold": round(macro_f1(y, pred_b3), 4)},
        rf_macro_f1_bootstrap_95ci=[round(float(np.percentile(rf_boot, 2.5)), 4),
                                    round(float(np.percentile(rf_boot, 97.5)), 4)],
        paired_rf_minus_b3={"observed": round(macro_f1(y, pred_rf) - macro_f1(y, pred_b3), 4),
                            "ci95": [round(lo, 4), round(hi, 4)],
                            "includes_zero": bool(lo <= 0 <= hi),
                            "p_diff_gt_0": round(float(np.mean(np.array(diff) > 0)), 4)},
        predictions={s: {"true": t, "RF": r, "B3": b}
                     for s, t, r, b in zip(df["season"], y, pred_rf, pred_b3)})


primary = {W: run_W(W, REALTIME_COLS) for W in DECISION_WEEKS}
secondary = {W: run_W(W, REALTIME_COLS, use_strain=True) for W in DECISION_WEEKS}

print("=== PRIMARY (real-time, ILI-only) macro-F1, pooled over 19 LOSO folds ===")
print(f"{'W':>3} {'RF':>7} {'B0':>7} {'B1':>7} {'B2':>7} {'B3':>7}  {'RF-B3 paired 95% CI':>24} incl0")
for W in DECISION_WEEKS:
    m = primary[W]["macro_f1"]; p = primary[W]["paired_rf_minus_b3"]
    print(f"{W:>3} {m['RF']:>7.4f} {m['B0_majority']:>7.4f} {m['B1_stratified_random_mean']:>7.4f} "
          f"{m['B2_climatology_threshold']:>7.4f} {m['B3_regression_then_threshold']:>7.4f}  "
          f"{p['observed']:+.4f} [{p['ci95'][0]:+.3f},{p['ci95'][1]:+.3f}]  {p['includes_zero']}")
print()
print("=== SECONDARY (retrospective, +strain) ===")
for W in DECISION_WEEKS:
    m = secondary[W]["macro_f1"]; p = secondary[W]["paired_rf_minus_b3"]
    print(f"  W={W:>2} RF={m['RF']:.4f}  B3={m['B3_regression_then_threshold']:.4f}  "
          f"paired {p['observed']:+.4f} [{p['ci95'][0]:+.3f},{p['ci95'][1]:+.3f}] incl0={p['includes_zero']}")


## D2, the regional severity figure

Continuous heatmap of the regional season peak, with national thresholds as **colorbar reference marks
only**. Cells are not colored or labeled by tier: CDC publishes no regional intensity thresholds, and
regional ILI has region-specific baselines, so a regional tier would be a label the thresholds do not
license.

Blocked until `data/raw/ILINet_regional.csv` exists (manual FluView download, region type
"HHS Regions"). The guard below records the block rather than skipping silently.

In [ ]:
# ---------------------------------------------------------------- D2 (guarded)
REGIONAL_PATH = DATA_DIR / "ILINet_regional.csv"
d2_status = {"built": False,
             "reason": f"{REGIONAL_PATH.name} not present in data/raw/; Step 0 of the plan is a "
                       f"manual FluView download with region type 'HHS Regions'. D2 is blocked, "
                       f"not skipped, and nothing about regional severity is asserted here."}
regional_summary = None
if REGIONAL_PATH.exists():
    reg = _load_ilinet(REGIONAL_PATH)
    assert reg["% WEIGHTED ILI"].notna().any(), "regional % WEIGHTED ILI is unpopulated; STOP"
    regs = sorted(reg["REGION"].dropna().unique())
    assert len(regs) == 10, f"expected 10 HHS regions, got {len(regs)}"
    rrows = []
    for (s, rg), g in reg[reg["season"].isin(COMPLETE)].groupby(["season", "REGION"]):
        rrows.append(dict(season=s, region=rg,
                          season_peak_max=round(float(g["% WEIGHTED ILI"].max()), 3)))
    regional_summary = pd.DataFrame(rrows)
    grid = regional_summary.pivot(index="region", columns="season", values="season_peak_max")
    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(grid.to_numpy(float), aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index, fontsize=8)
    cb = fig.colorbar(im, ax=ax)
    cb.set_label("Regional season peak % weighted ILI\n(ticks: national reference thresholds)")
    for k, v in CDC_ILI_IT.items():
        cb.ax.axhline(v, color="black", lw=0.6)
        cb.ax.text(1.6, v, k, va="center", fontsize=6)
    ax.set_title("D2: regional season peak ILI. Continuous scale; national reference thresholds "
                 "only, not regional tiers.", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "12_D2_severity_heatmap.png", dpi=150); plt.close(fig)
    d2_status = {"built": True, "regions": len(regs)}


## Persisted results and the pre-committed adjudication

The plan pre-committed to reporting a negative if the RF did not beat B3. The adjudication below is
applied to whatever the numbers turned out to be, not chosen after seeing them.

In [ ]:
# ---------------------------------------------------------------- persist
def to_md(df):
    cols = list(df.columns)
    return "\n".join(["| " + " | ".join(cols) + " |",
                      "| " + " | ".join("---" for _ in cols) + " |"] +
                     ["| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |"
                      for r in df.itertuples(index=False)])


tier_payload = {
    "status": "target definition, pending Joshua's checkpoint-2 review; no advisor sign-off on the "
              "concrete numbers, only on the method (Dr. Mitra, 2026-08-02)",
    "framework": "ILI-only approximation of CDC's season severity framework using CDC's published "
                 "ILI intensity thresholds. NOT CDC's official season severity classification, "
                 "which is a 2-of-3 indicator vote over ILI, hospitalization rate, and P&I mortality.",
    "thresholds": CDC_ILI_IT, "threshold_source": CDC_IT_SOURCE,
    "threshold_reference_seasons": CDC_IT_REFERENCE_SEASONS, "threshold_vintage": CDC_IT_VINTAGE_NOTE,
    "classification_statistic": "season peak weekly raw % WEIGHTED ILI (season_peak_max). This is "
                                "what CDC classifies on. The geometric mean of the three highest "
                                "weeks is how the thresholds were ESTIMATED, not how a season is "
                                "classified; season_peak_gm3 is retained only to document that.",
    "labels": ["Low", "Moderate", "High", "Very High"],
    "very_high_empty": "No season in 22 reaches IT98=8.6. The class is empty by observation and is "
                       "dropped from the classifier, not silently omitted.",
    "not_assessed": NOT_ASSESSED,
    "tier_counts_modeling_19": model_tbl["tier_max"].value_counts().to_dict(),
    "tier_counts_assessed_21": assessed["tier_max"].value_counts().to_dict(),
    "boundary_sensitive": model_tbl.loc[model_tbl["tier_boundary_sensitive"], "season"].tolist(),
    "holiday_artifact_tension": "CDC's statistic is the unsmoothed weekly maximum. This project's "
                                "notebook 02 finding is that the unsmoothed maximum is inflated by "
                                "the wk52 reporting artifact, which is why the project smooths. The "
                                "CDC-anchored tier is therefore holiday-inflated for the seasons "
                                "flagged holiday_shift; tier_smoothed is the mandatory sensitivity.",
    "validation": {"overall": f"{agree_all}/{n_all}", "comparable_only": f"{agree_comp}/{n_comp}",
                   "reporting_rule": "always report overall first; quoting only the comparable-only "
                                     "figure would overstate agreement, because the two excluded "
                                     "seasons are two of the three disagreements",
                   "disagreements_overall": val.loc[~val["agrees"], "season"].tolist(),
                   "non_comparable": NON_COMPARABLE, "table": val.to_dict("records")},
    "season_table": season_table.to_dict("records"),
}
(RESULTS_DIR / "08_tier_validation.json").write_text(json.dumps(tier_payload, indent=2), encoding="utf-8")

clf_payload = {
    "status": "preliminary; underpowered by design and reported as such",
    "target": "tier_max on the 19-season modeling set",
    "classes": CLASSES, "class_counts": primary[8]["class_counts"],
    "sample_size_caveat": "n=19 with 3 classes and 6 feature columns. The Low class has 4 members, "
                          "so some LOSO folds train on 3 examples of it. Overfitting is expected.",
    "rf_params": {k: (v if not isinstance(v, type(None)) else None) for k, v in RF_PARAMS.items()},
    "pre_registered": "hyperparameters, seed and scoring rule fixed before any result was seen; "
                      "no post-hoc tuning was performed",
    "scoring": "each LOSO fold predicts one held-out season; predictions pooled over all 19 folds "
               "and macro-F1 computed once over the pool. A per-fold average is undefined at n=1.",
    "macro_f1_status": "descriptive only, pre-registered as such; read the confusion matrix first",
    "primary_realtime": {str(W): primary[W] for W in DECISION_WEEKS},
    "secondary_retrospective": {str(W): secondary[W] for W in DECISION_WEEKS},
    "feature_sets": {"primary": REALTIME_COLS + ["(near-real-time under the index-cutoff convention)"],
                     "secondary": REALTIME_COLS + ["dominant_strain (reporting-lagged, retrospective)"]},
    "firewall": {"index_firewall": f"PASS across {audit_pairs} (season,W) pairs; every feature "
                                   f"index <= W",
                 "label_note": "the LABEL derives from the full-season peak by design, as in 05 "
                               "and 06 where the target is also the realized season peak; the "
                               "firewall governs features, not the target"},
    "d2": d2_status,
}

# ------- Pre-committed adjudication against B3, written before the numbers were interpreted -------
_p = {W: primary[W]["paired_rf_minus_b3"] for W in DECISION_WEEKS}
_excl = [W for W in DECISION_WEEKS if not _p[W]["includes_zero"]]
clf_payload["conclusion"] = {
    "verdict": "No demonstrated advantage over B3, the regression-then-threshold baseline.",
    "reasoning": [
        f"Paired bootstrap intervals for RF minus B3 include zero at W=12 ({_p[12]['ci95']}) and "
        f"W=16 ({_p[16]['ci95']}). Only W=8 excludes zero, and only marginally "
        f"({_p[8]['ci95'][0]:+.3f} lower bound).",
        "The pattern runs backwards. The apparent advantage is largest at W=8, the earliest and "
        "least informative decision week, and shrinks to +0.06 by W=16. A real skill advantage "
        "should not decay as more of the season becomes observable.",
        f"Macro-F1 is itself non-monotonic in W (RF {primary[8]['macro_f1']['RF']} at W=8 versus "
        f"{primary[12]['macro_f1']['RF']} at W=12). More information making the model worse is a "
        "noise signature, not a skill signature.",
        "Three W values were inspected. One marginal exclusion out of three comparisons is what "
        "chance produces; it is descriptive, never confirmation.",
    ],
    "w_intervals_excluding_zero": _excl,
    "multiple_comparison_caveat": "three W values inspected; descriptive, never confirmation",
    "consistent_with": "notebook 05's negative point-forecast result and notebook 07's unsupported "
                       "H1. The classifier is reported as a negative, per the pre-commitment in "
                       "PLAN-severity-tiers.md Step 5.",
}
# Direction of the raw-versus-smoothed tier disagreements, which tests the holiday-inflation claim.
_dis = season_table[season_table["tier_max"] != season_table["tier_smoothed"]]
_rank = {"Low": 0, "Moderate": 1, "High": 2, "Very High": 3}
tier_payload["raw_vs_smoothed_disagreements"] = {
    "seasons": _dis["season"].tolist(),
    "all_raw_tier_higher": bool(all(_rank[r["tier_max"]] > _rank[r["tier_smoothed"]]
                                    for _, r in _dis.iterrows())),
    "interpretation": "In every season where the two statistics disagree, the unsmoothed CDC "
                      "statistic tiers HIGHER than the project's smoothed statistic. That is the "
                      "direction the notebook 02 holiday-artifact finding predicts, and it is why "
                      "tier_smoothed is a mandatory sensitivity rather than an optional one.",
}
(RESULTS_DIR / "08_tier_validation.json").write_text(json.dumps(tier_payload, indent=2), encoding="utf-8")
(RESULTS_DIR / "08_severity_classifier.json").write_text(json.dumps(clf_payload, indent=2), encoding="utf-8")

# ---------------------------------------------------------------- markdown companions
md = ["# 08 severity tiers: CDC-anchored definition and validation", "",
      "**ILI-only approximation of CDC's severity framework**, using CDC's published ILI intensity",
      "thresholds. This is NOT CDC's official season severity classification, which is a 2-of-3",
      "indicator vote over ILI, hospitalization rate, and pneumonia-and-influenza mortality.", "",
      f"Thresholds: IT50={CDC_ILI_IT['IT50']}, IT90={CDC_ILI_IT['IT90']}, IT98={CDC_ILI_IT['IT98']}.",
      f"Source: {CDC_IT_SOURCE}", f"Reference seasons: {CDC_IT_REFERENCE_SEASONS}", "",
      f"Vintage: {CDC_IT_VINTAGE_NOTE}", "",
      "Classification statistic: season peak weekly raw % WEIGHTED ILI. The geometric mean of the",
      "three highest weeks is how the thresholds were ESTIMATED, not how a season is classified.", "",
      "## Tier counts", "",
      f"- Modeling set (19): {model_tbl['tier_max'].value_counts().to_dict()}",
      f"- Assessed (21, excludes 2020-21): {assessed['tier_max'].value_counts().to_dict()}",
      "- Very High: 0 seasons in 22. No season reaches IT98=8.6.", "",
      "2020-21 carries no CDC-anchored tier. CDC did not assess it.", "",
      "## Validation against CDC's published classifications", "",
      f"**{agree_all}/{n_all} overall**, and **{agree_comp}/{n_comp} after excluding the two",
      "pre-declared pandemic and pandemic-adjacent seasons as non-comparable.** Both are reported",
      "together: quoting only the second would overstate agreement, because the two excluded",
      "seasons are two of the three disagreements.", "",
      to_md(val[["season", "cdc_published", "ili_only_tier", "season_peak_max", "agrees",
                 "comparable"]]), "",
      "## Season table", "",
      to_md(season_table[["season", "season_peak_max", "peak_ili_pct", "tier_assigned",
                          "tier_smoothed", "tier_gm3", "tier_margin", "tier_boundary_sensitive",
                          "excluded_from_modeling"]]), "",
      "## Raw versus smoothed disagreements", "",
      tier_payload["raw_vs_smoothed_disagreements"]["interpretation"],
      f"Seasons: {', '.join(_dis['season'].tolist())}.", ""]
(RESULTS_DIR / "08_tier_validation.md").write_text("\n".join(md), encoding="utf-8")

cm_rows = []
for W in DECISION_WEEKS:
    m = primary[W]["macro_f1"]; p = primary[W]["paired_rf_minus_b3"]
    cm_rows.append(dict(W=W, RF=m["RF"], B0=m["B0_majority"], B1=m["B1_stratified_random_mean"],
                        B2=m["B2_climatology_threshold"], B3=m["B3_regression_then_threshold"],
                        RF_minus_B3=p["observed"], CI95=str(p["ci95"]),
                        includes_zero=p["includes_zero"]))
md2 = ["# 08 severity classifier (3-class, LOSO, n=19)", "",
       f"**{clf_payload['conclusion']['verdict']}**", ""]
md2 += [f"- {r}" for r in clf_payload["conclusion"]["reasoning"]]
md2 += ["", f"Class counts: {primary[8]['class_counts']}. Very High dropped: empty by observation.",
        "", clf_payload["sample_size_caveat"], "",
        "## Primary, real-time (ILI-only) macro-F1, pooled over 19 LOSO folds", "",
        to_md(pd.DataFrame(cm_rows)), "",
        "Macro-F1 is pre-registered as descriptive only. Read the confusion matrix first.", ""]
for W in DECISION_WEEKS:
    cmx = primary[W]["confusion_matrix"]["rows_true_cols_pred"]
    md2 += [f"### W={W} confusion matrix (rows true, cols predicted: {CLASSES})", "",
            to_md(pd.DataFrame(cmx, index=CLASSES, columns=CLASSES).reset_index(names="true")), "",
            f"accuracy {primary[W]['accuracy']}, per-class F1 {primary[W]['per_class_f1']}", ""]
md2 += ["## Secondary, retrospective (+dominant strain)", "",
        "Strain is reporting-lagged, so this panel is explanatory and is never a forecast.", ""]
md2 += [f"- W={W}: RF {secondary[W]['macro_f1']['RF']}, paired vs B3 "
        f"{secondary[W]['paired_rf_minus_b3']['observed']:+.4f} "
        f"{secondary[W]['paired_rf_minus_b3']['ci95']}, includes zero "
        f"{secondary[W]['paired_rf_minus_b3']['includes_zero']}" for W in DECISION_WEEKS]
md2 += ["", f"## Firewall", "", clf_payload["firewall"]["index_firewall"], "",
        clf_payload["firewall"]["label_note"], "", "## D2", "",
        ("Built." if d2_status["built"] else f"NOT BUILT. {d2_status['reason']}"), ""]
(RESULTS_DIR / "08_severity_classifier.md").write_text("\n".join(md2), encoding="utf-8")
print()
print("=== persisted ===", [p.name for p in sorted(RESULTS_DIR.glob('08_*'))])
print("=== D2 ===", d2_status)
print()
print("=== season table (assigned tiers) ===")
print(to_md(season_table[["season", "season_peak_max", "peak_ili_pct", "tier_assigned",
                          "tier_smoothed", "tier_margin", "tier_boundary_sensitive",
                          "excluded_from_modeling"]]))


## Conclusion

The tier definition is implemented and validated: **9/12 agreement with CDC's published
classifications overall, 9/10 among comparable seasons**, with the single remaining disagreement
(2014-15) attributable to the 2-of-3 framework difference this project cannot reproduce.

The classifier is a **negative result**. It does not demonstrate an advantage over thresholding a
one-variable regression, and the pattern of its apparent edge (largest at the earliest decision week,
decaying as more of the season becomes observable, non-monotonic in W) is a noise signature. That is
reported plainly, consistent with 05's negative point-forecast result and 07's unsupported H1.

**Still outstanding:** D2 is blocked on the regional ILINet download, and the tier definition needs
Joshua's checkpoint-2 review before anything here is committed.